# Step 2 — 00. Checkpoint registration

목적은 공개 checkpoint의 출처, 로컬 SHA-256, 전처리, backbone,
학습 데이터 표기, 512D 출력과 Grad-CAM target layer를 하나의
불변 `ModelSpec`으로 등록하는 것입니다.

이 노트북은 모델을 학습하지 않습니다. 모델별 공식 구현·전처리·target
layer와 공통 crop의 RGB source order는 Step 2 설정의 **profile** 단위로
자동 선택합니다. 사용자는 실제 checkpoint 경로만 지정합니다.
기존 manifest는 덮어쓰지 않습니다.

## Profile 구조

기존 `MODEL_NAME` (family 단위) 대신 `MODEL_PROFILE` (checkpoint 단위)을
사용합니다. 이를 통해 동일 family에서 학습 데이터가 다른 checkpoint를
독립적으로 등록·관리할 수 있습니다.

In [ ]:
# cell 1 : 환경 설정 및 profile 선택
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("research와 configs가 있는 프로젝트 루트를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

# ─── Profile 선택 ──────────────────────────────────────────────────
MODEL_PROFILE = "arcface_ms1mv3_r100"  # 아래 available profiles에서 선택
# ──────────────────────────────────────────────────────────────────

MODE = "dev"               # 빠른 검증은 "dev", 전체 논문 실행만 "real"
DATA_FRACTION = 1          # identity 단위 사용 비율; 0 < 값 <= 1
SEED = 42                  # 부분집합·tie-break·random control 재현 seed
EXECUTE_STAGE = False
WRITE_OUTPUTS = False

# Profile 유효성 검증
all_profiles = (
    CONFIG["models"]["selected_profiles"]
    + CONFIG["models"].get("bridge_profiles", [])
)
available_profiles = CONFIG["models"]["profiles"]
blocked_profiles = CONFIG["models"].get("blocked_profiles", [])

if MODEL_PROFILE in blocked_profiles:
    blocked_reason = available_profiles[MODEL_PROFILE].get(
        "blocked_reason", "차단된 profile입니다."
    )
    raise RuntimeError(
        f"차단된 profile입니다: {MODEL_PROFILE}\n{blocked_reason}"
    )

if MODEL_PROFILE not in available_profiles:
    raise ValueError(
        f"지원하지 않는 MODEL_PROFILE: {MODEL_PROFILE}\n"
        f"사용 가능한 profile: {sorted(available_profiles)}"
    )

if MODEL_PROFILE not in all_profiles:
    print(
        f"⚠ WARNING: {MODEL_PROFILE}은 selected_profiles나 "
        "bridge_profiles에 포함되어 있지 않습니다."
    )

PROFILE = available_profiles[MODEL_PROFILE]
MODEL_FAMILY = PROFILE["family"]

if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")

print(f"Profile: {MODEL_PROFILE}")
print(f"Family: {MODEL_FAMILY}")
print(f"Role: {PROFILE['role']}")
print(f"Training dataset: {PROFILE['training_dataset']}")
print(f"Architecture: {PROFILE['architecture']}")

In [ ]:
# cell 2 : profile 메타데이터 검증 및 checkpoint 경로 설정
from research.embeddings import (
    CheckpointProvenance,
    ModelSpec,
    PreprocessingSpec,
    write_model_spec,
    download_checkpoint,
    resolve_checkpoint_path,
)

# ─── Profile 메타데이터 불일치 차단 ────────────────────────────────
EXPECTED_PROFILE_RULES = {
    "arcface_ms1mv3_r100": ("arcface", "ms1mv3", "iresnet100"),
    "adaface_ms1mv3_r100": ("adaface", "ms1mv3", "ir_101"),
    "adaface_ms1mv2_r100": ("adaface", "ms1mv2", "ir_101"),
    "magface_ms1mv2_iresnet100": ("magface", "ms1mv2", "iresnet100"),
}

if MODEL_PROFILE in EXPECTED_PROFILE_RULES:
    expected = EXPECTED_PROFILE_RULES[MODEL_PROFILE]
    actual = (
        PROFILE["family"],
        PROFILE["training_dataset"],
        PROFILE["architecture"],
    )
    if actual != expected:
        raise RuntimeError(
            f"Profile metadata mismatch: expected={expected}, actual={actual}"
        )

# ─── Checkpoint 경로 설정 ──────────────────────────────────────────
# 사용자가 직접 경로를 지정하려면 아래 CHECKPOINT_PATH에 값을 설정합니다.
# None이면 기본 경로(models/<family>/<filename>)를 사용합니다.
CHECKPOINT_PATH = None  # 예: PROJECT_ROOT / "models/arcface/ms1mv3_r100_backbone.pth"

CHECKPOINT_SOURCE_URL = (
    PROFILE.get("checkpoint_source_url")
    or PROFILE.get("checkpoint_source_page")
)

# 모델별 구현 계약은 설정에서 자동 선택하며 이 셀에서 임의로 바꾸지 않는다.
SOURCE_COLOR_ORDER = CONFIG["aligned_crops"]["source_color_order"]
IMPLEMENTATION_REPOSITORY = PROFILE["implementation_repository"]
MODULE_FACTORY = PROFILE["loader_factory"]
TARGET_LAYER = PROFILE["target_layer"]
MODEL_COLOR_ORDER = PROFILE["preprocessing"]["model_color_order"]
CHANNEL_MEAN = tuple(PROFILE["preprocessing"]["mean"])
CHANNEL_STD = tuple(PROFILE["preprocessing"]["std"])

print(f"Color order: source={SOURCE_COLOR_ORDER}, model={MODEL_COLOR_ORDER}")
print(f"Mean: {CHANNEL_MEAN}")
print(f"Std: {CHANNEL_STD}")
print(f"Target layer: {TARGET_LAYER}")
print(f"Loader factory: {MODULE_FACTORY}")

In [ ]:
# cell 3 : checkpoint 다운로드 (필요 시)
if EXECUTE_STAGE:
    resolved_path = resolve_checkpoint_path(
        PROJECT_ROOT, MODEL_PROFILE, explicit_path=CHECKPOINT_PATH
    )
    if not resolved_path.is_file():
        print(f"Checkpoint가 존재하지 않습니다: {resolved_path}")
        print("자동 다운로드를 시도합니다...")
        resolved_path = download_checkpoint(
            PROJECT_ROOT,
            MODEL_PROFILE,
            PROFILE,
            explicit_path=CHECKPOINT_PATH,
        )
    CHECKPOINT_PATH = resolved_path
    print(f"✓ Checkpoint path: {CHECKPOINT_PATH}")
    print(f"  File size: {CHECKPOINT_PATH.stat().st_size:,} bytes")
else:
    print("EXECUTE_STAGE=False: checkpoint 다운로드를 건너뜁니다.")

In [ ]:
# cell 4 : ModelSpec 등록
required = {
    "CHECKPOINT_PATH": CHECKPOINT_PATH,
    "CHECKPOINT_SOURCE_URL": CHECKPOINT_SOURCE_URL,
}

if EXECUTE_STAGE:
    missing = [name for name, value in required.items() if value is None]
    if missing:
        raise RuntimeError(f"checkpoint 등록값이 비어 있습니다: {missing}")
    checkpoint = CheckpointProvenance.from_file(
        CHECKPOINT_PATH,
        source_url=CHECKPOINT_SOURCE_URL,
    )
    preprocessing = PreprocessingSpec(
        input_height=PROFILE["preprocessing"]["input_size"][0],
        input_width=PROFILE["preprocessing"]["input_size"][1],
        source_color_order=SOURCE_COLOR_ORDER,
        model_color_order=MODEL_COLOR_ORDER,
        channel_mean=tuple(CHANNEL_MEAN),
        channel_std=tuple(CHANNEL_STD),
    )
    spec = ModelSpec(
        family=PROFILE["family"],
        architecture=PROFILE["architecture"],
        training_dataset=PROFILE["training_dataset"],
        implementation_repository=IMPLEMENTATION_REPOSITORY,
        checkpoint=checkpoint,
        preprocessing=preprocessing,
        target_layer=TARGET_LAYER,
        embedding_dim=PROFILE["embedding_dim"],
        module_factory=MODULE_FACTORY,
    )
    registration = spec.to_manifest()
    # profile 수준 메타데이터 추가 (model_uid 해시에는 포함되지 않음)
    registration["profile_id"] = MODEL_PROFILE
    registration["analysis_role"] = PROFILE["role"]
    registration["run_compression"] = PROFILE["run_compression"]
    registration["run_gradcam"] = PROFILE["run_gradcam"]
    if WRITE_OUTPUTS:
        destination = (
            PROJECT_ROOT
            / "runs/step2/model_registry"
            / f"{spec.model_uid}.json"
        )
        write_model_spec(destination, spec)
        registration["written_to"] = str(destination)
else:
    registration = {
        "status": "not_executed",
        "reason": "EXECUTE_STAGE=False",
        "profile_id": MODEL_PROFILE,
        "family": MODEL_FAMILY,
        "automatic_contract": {
            "architecture": PROFILE["architecture"],
            "training_dataset": PROFILE["training_dataset"],
            "loader_factory": MODULE_FACTORY,
            "target_layer": TARGET_LAYER,
            "model_color_order": MODEL_COLOR_ORDER,
            "mean": CHANNEL_MEAN,
            "std": CHANNEL_STD,
        },
    }
registration

다음 단계는 `01_preprocessing_and_model_smoke.ipynb`입니다. 새
checkpoint나 전처리 값으로 바꾸면 기존 manifest를 수정하지 말고 새
`model_uid`로 다시 등록합니다.

## 사용 가능한 Profile 목록

| Profile ID | Family | Dataset | Role |
|---|---|---|---|
| `arcface_ms1mv3_r100` | arcface | MS1MV3 | primary |
| `adaface_ms1mv3_r100` | adaface | MS1MV3 | primary |
| `magface_ms1mv2_iresnet100` | magface | MS1MV2 | primary |
| `adaface_ms1mv2_r100` | adaface | MS1MV2 | bridge |
| `arcface_ms1mv2_r100` | arcface | MS1MV2 | ❌ blocked |